In [1]:
# imports

# handle requests 
import requests 

# db adapter
import psycopg2

# dict flattening functions
from src.flattening_data import (
    flatten_repo_dictionaries,
    flatten_repo_commit_dictionaries,
    flatten_repo_issues_dictionaries,
    normalize_data_columns
)

# get data from GitHub REST API
from src.api_helpers import (
    get_data
)

# load data into the db tables
from src.db_helpers import (
    table_insertion,
    retrieve_last_user_id
)

# import pipeline variables
from src.config import (
    TOKEN,
    DB_NAME,
    PASSWORD,
    headers,
    rename_users_map,
    rename_repos_map,
    rename_commits_map,
    rename_issues_map
)

In [2]:
# ensure env vars exists
print("TOKEN exists:", TOKEN is not None)
print("DB_NAME exists:", DB_NAME is not None)
print("PASSWORD exists:", PASSWORD is not None)
print("headers exists:", headers is not None)
print("rename_users_map exists:", rename_users_map is not None)
print("rename_repos_map exists:", rename_repos_map is not None)
print("rename_commits_map exists:", rename_commits_map is not None)
print("rename_issues_map exists:", rename_issues_map is not None)
print("most_recent_user_id exists:", rename_issues_map is not None)

TOKEN exists: True
DB_NAME exists: True
PASSWORD exists: True
headers exists: True
rename_users_map exists: True
rename_repos_map exists: True
rename_commits_map exists: True
rename_issues_map exists: True
most_recent_user_id exists: True


In [3]:
try:
    # pipeline vars

    # get most recent user id to get data from after that point
    most_recent_user_id = retrieve_last_user_id(DB_NAME, PASSWORD)

    # number of pages to ingest
    pages = 3

    # pagination size
    users_per_page = 20
    repos_per_page = 10
    commits_per_page = 20
    issues_per_page = 20
    
    # store user/repos/commits/issues data
    all_users_data = []
    all_repos_data = []
    all_commits_data = []
    all_issues_data = []

    # for each page get x users
    for i in range(pages):

        # get users data
        endpoint = f"/users?since={most_recent_user_id}&per_page={users_per_page}"

        # returns list
        users = get_data(endpoint, headers)

        # break if users is empty list > stop pagination loop
        if not users:
            break

        # get the usernames to get more data
        logins = [user["login"] for user in users]

        # for each user get more detailed data
        for login in logins:
            
            # endpoint for specific user data using username
            endpoint = f"/users/{login}"

            # returns a dict per user
            user_data = get_data(endpoint, headers)

            # rename user data cols
            normalize_data_columns(user_data, rename_users_map)

            # append user dict to all data
            all_users_data.append(user_data)

            # repo section
            
            # endpoint for specific user repos data using username
            endpoint = f"/users/{login}/repos?per_page={repos_per_page}"

            # returns a list
            repo_data = get_data(endpoint, headers)
            
            # for each repo get the commits/issues for that repo
            for repo in repo_data:

                # flatten repo dictionaries
                flatten_repo_dictionaries(repo)

                # get repo name for url
                repo_name = repo["name"]

                # get repo id for flattening commits/issues dicts
                repo_id = repo['id']

                # rename repo data cols
                normalize_data_columns(repo, rename_repos_map)

                # endpoint for getting commits of a user's specific repo
                endpoint = f"/repos/{login}/{repo_name}/commits?per_page={commits_per_page}"

                # returns a list
                commit_data = get_data(endpoint, headers)

                # flatten the dictionaries in each commit
                for row in commit_data:
            
                    flatten_repo_commit_dictionaries(row, repo_id)

                    # rename commits data cols
                    normalize_data_columns(row, rename_commits_map)

                all_commits_data.extend(commit_data)

                # this section is to ingest issues data
                
                # if repo has issues then call
                if repo["has_issues"] == True:
                    
                    # # endpoint for getting issues of a user's specific repo
                    endpoint = f"/repos/{login}/{repo_name}/issues?per_page={issues_per_page}"

                    # returns list
                    issue_data = get_data(endpoint, headers)

                    # flatten the dictionaries in each issue 
                    for row in issue_data:
                        
                        flatten_repo_issues_dictionaries(row, repo_id)

                        # rename commits data cols
                        normalize_data_columns(row, rename_issues_map)
    
                    # add issue data to all issues data
                    all_issues_data.extend(issue_data)

            # extend repos data list to include repo data
            all_repos_data.extend(repo_data)
        
        # GitHub REST api with users endpoint uses 'since' to get the users where the id is after 'most_recent_user_id'
        # therefore, get the last id of the current call to get the next page
        most_recent_user_id = users[-1]["id"]

# handle HTTP status errors first, then catch other request failures
except requests.exceptions.HTTPError as e:
    print("HTTP error:", e)
    
except requests.exceptions.RequestException as e:
    print("Other request error:", e)

In [4]:
# data size
print("Number of users:", len(all_users_data))
print("Number of repos:",len(all_repos_data))
print("Number of commits:", len(all_commits_data))
print("Number of issues:", len(all_issues_data))

Number of users: 60
Number of repos: 530
Number of commits: 7643
Number of issues: 423


In [5]:
# connect to postgres DB
conn = psycopg2.connect(f"dbname={DB_NAME} user=postgres password={PASSWORD}")

# insert data into tables
table_insertion(conn, 'users', all_users_data)
table_insertion(conn, 'repos', all_repos_data)
table_insertion(conn, 'commits', all_commits_data)
table_insertion(conn, 'issues', all_issues_data)

# close connection
conn.close()